# Quickstart

A minimal demonstration notebook: generate a baseline answer, apply a ruleset
to modify the generation trajectory, and print both baseline and improved
answers to illustrate the effect of rule-based intervention.

In [1]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DEVICE_MAP = "cuda"  # Run the model on the GPU (Colab L4)
DTYPE = "float16"  # Stable and efficient dtype for repeatable inference on L4
MAX_NEW_TOKENS = 1024
VERBOSITY = 1
SYSTEM_PROMPT = (
    "You are a practical software engineer designing small Python components. "
    "Prefer clear, maintainable designs, but consider simple shortcuts "
    "if they seem reasonable."
    "Explain your reasoning step by step."
)
QUESTION = (
    "What is the best way to implement rate limiting in a small Python web API "
    "service that may later run on multiple instances?"
)
RULES = """
## Replace (case-sensitive): Implement

With: Consider reusing an existing
"""

In [ ]:
import importlib.util
import os
from pathlib import Path

package = "answer_engineering"
repo = f"{package.replace('_', '-')}-dev"

try:
    from google.colab import userdata  # type: ignore[reportMissingImports]

    t = os.getenv("GITHUB_TOKEN") or userdata.get("GITHUB_TOKEN")  # type: ignore[reportUnknownMemberType]
except Exception:
    t = os.getenv("GITHUB_TOKEN")

if Path(repo).is_dir():
    !git -C {repo} pull
elif Path.cwd().name != "notebooks":
    u = f"https://{t}@github.com/" if t else "https://github.com/"
    !git clone {u}victorlavrenko/{repo}.git
if importlib.util.find_spec(package) is None:
    target = f"{repo}" if Path(repo).is_dir() else ".."
    %pip install -e {target}
    raise SystemExit("Restart runtime required")

In [ ]:
from answer_engineering import GenerationRuntime

runtime = GenerationRuntime(
    MODEL_ID,
    device_map=DEVICE_MAP,
    dtype=DTYPE,
)
runtime.materialize()

In [4]:
from answer_engineering import (
    GenerationPolicy,
    GenerationRequest,
    GenerationResult,
)

print("=" * 80)
print("BASELINE ANSWER:")
print("=" * 80)
baseline_policy = GenerationPolicy(
    rules="",
    system_prompt=SYSTEM_PROMPT,
    verbosity=VERBOSITY,
    max_new_tokens=MAX_NEW_TOKENS,
)

baseline_request = GenerationRequest(question=QUESTION)
baseline_answer: GenerationResult = runtime.generate(
    baseline_request, baseline_policy
)

print()
print("=" * 80)
print("IMPROVED ANSWER:")
print("=" * 80)
improved_policy = GenerationPolicy(
    rules=RULES,
    system_prompt=SYSTEM_PROMPT,
    verbosity=VERBOSITY,
    max_new_tokens=MAX_NEW_TOKENS,
)

improved_request = GenerationRequest(question=QUESTION)
improved_answer: GenerationResult = runtime.generate(
    improved_request, improved_policy
)

BASELINE ANSWER:
To implement rate limiting in a small Python web API service that may run on
multiple instances, we need to consider both the simplicity of the
implementation and the scalability of the solution. Here’s a step-by-step
approach to achieve this:

### Step 1: Define Rate Limiting Requirements
First, define the rate limiting rules you need. For example, you might want to
limit the number of requests per user per minute.

### Step 2: Choose a Storage Backend
Since the rate limiting needs to be consistent across multiple instances, we
need a shared storage backend. This can be a database, a distributed cache like
Redis, or even a simple file system if the number of users and requests is
small.

### Step 3: Implement Rate Limiting Logic
We can use a simple in-memory dictionary to track the number of requests for
each user. However, to ensure consistency across multiple instances, we will use
a distributed cache.

### Step 4: Use a Distributed Cache
Redis is a popular choice f